# 🏷️ EDA — Noms de famille (`1_noms_clean.json`)

Exploration des données brutes de noms de famille.  
Fichier source : `noms/data/1_noms_clean.json`

**Plan :**
1. Chargement & aperçu général
2. Couverture des champs
3. Analyse des noms (longueur, initiales, terminaisons)
4. Analyse textuelle de `origine_brute`
5. Bag-of-Words sur `origine_brute` et `origine` (lemmatisée)
6. Détection des langues / origines géographiques
7. Analyse du graphe de noms liés (`noms_lies`)
8. Analyse des groupes (`id_groupe`)
9. Word Cloud
10. Récapitulatif

In [ ]:
import json
import re
import os
from collections import Counter, defaultdict
from pathlib import Path
from itertools import islice

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

DATA_PATH = Path('data/1_noms_clean.json')
print('Fichier :', DATA_PATH.resolve())

## 1. Chargement & aperçu général

In [ ]:
with open(DATA_PATH, encoding='utf-8') as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
print(f'Nombre de noms : {len(df):,}')
print(f'Colonnes       : {list(df.columns)}')
df.head(3)

In [ ]:
df.info()

## 2. Couverture des champs

In [ ]:
total = len(df)

def is_filled(val):
    if val is None:
        return False
    if isinstance(val, list):
        return len(val) > 0
    if isinstance(val, str):
        return bool(val.strip())
    return bool(val)

coverage = {col: df[col].apply(is_filled).sum() for col in df.columns}
cov_df = pd.DataFrame({'champ': list(coverage.keys()),
                        'n_rempli': list(coverage.values())})
cov_df['pct'] = cov_df['n_rempli'] / total * 100

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(cov_df['champ'], cov_df['pct'],
               color=sns.color_palette('Blues_r', len(cov_df)))
ax.set_xlabel('% rempli')
ax.set_title('Couverture des champs', fontsize=14, fontweight='bold')
ax.axvline(100, color='gray', linestyle='--', alpha=0.5)
for bar, pct in zip(bars, cov_df['pct']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 3. Analyse des noms (longueur, initiales, terminaisons)

In [ ]:
df['longueur']  = df['nom'].str.len()
df['initiale']  = df['nom'].str[0].str.upper()
df['suffix2']   = df['nom'].str[-2:]
df['suffix3']   = df['nom'].str[-3:]
df['nb_tokens'] = df['nom'].str.split().str.len()

print('Statistiques sur la longueur des noms :')
print(df['longueur'].describe().round(2))
print(f'\nNom le plus court : {df.loc[df.longueur.idxmin(), "nom"]} ({df.longueur.min()} chars)')
print(f'Nom le plus long  : {df.loc[df.longueur.idxmax(), "nom"]} ({df.longueur.max()} chars)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution longueurs
axes[0].hist(df['longueur'], bins=range(1, 25), color='#5b8fd6', edgecolor='none', alpha=0.85)
axes[0].axvline(df['longueur'].median(), color='orange', linestyle='--',
                linewidth=2, label=f'Médiane = {df.longueur.median():.0f}')
axes[0].axvline(df['longueur'].mean(), color='red', linestyle=':',
                linewidth=2, label=f'Moyenne = {df.longueur.mean():.1f}')
axes[0].set_title('Distribution des longueurs de noms', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nombre de caractères')
axes[0].set_ylabel('Nombre de noms')
axes[0].legend()

# Initiales
init_counts = df['initiale'].value_counts().sort_index()
axes[1].bar(init_counts.index, init_counts.values,
            color=sns.color_palette('viridis', len(init_counts)))
axes[1].set_title('Fréquence des initiales', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Lettre')
axes[1].set_ylabel('Nombre de noms')
axes[1].tick_params(axis='x', labelsize=7)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, col, color, label in zip(
    axes,
    ['suffix2', 'suffix3'],
    ['#7bcfa0', '#a07bcf'],
    ['2 caractères', '3 caractères']
):
    top = df[col].value_counts().head(20)
    ax.barh(top.index[::-1], top.values[::-1], color=color, alpha=0.85)
    ax.set_title(f'Top 20 terminaisons ({label})', fontsize=12, fontweight='bold')
    ax.set_xlabel('Fréquence')
    for i, v in enumerate(top.values[::-1]):
        ax.text(v + 5, i, f'{v:,}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Préfixes (2-3 premiers caractères)
df['prefix2'] = df['nom'].str[:2]
df['prefix3'] = df['nom'].str[:3]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, col, color in zip(axes, ['prefix2', 'prefix3'], ['#f0a060', '#60a0f0']):
    top = df[col].value_counts().head(20)
    ax.barh(top.index[::-1], top.values[::-1], color=color, alpha=0.85)
    ax.set_title(f'Top 20 préfixes ({col[-1]} chars)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Fréquence')

plt.tight_layout()
plt.show()

## 4. Analyse textuelle de `origine_brute`

In [ ]:
df['nb_mots_brute'] = df['origine_brute'].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

print('Statistiques sur le nombre de mots dans origine_brute :')
print(df['nb_mots_brute'].describe().round(2))

# Noms sans texte d'origine
empty = (df['nb_mots_brute'] == 0).sum()
print(f'\nNoms sans texte : {empty:,} ({100*empty/total:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
vals = df[df.nb_mots_brute > 0]['nb_mots_brute']
axes[0].hist(vals, bins=50, color='#5b8fd6', edgecolor='none', alpha=0.85)
axes[0].axvline(vals.median(), color='orange', linestyle='--',
                linewidth=2, label=f'Médiane = {vals.median():.0f}')
axes[0].set_title('Distribution du nb de mots (origine_brute, entrées non vides)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nombre de mots')
axes[0].set_ylabel('Nombre de noms')
axes[0].legend()

# Catégories de richesse
bins_labels = ['0', '1-10', '11-30', '31-60', '61-100', '100+']
cuts = pd.cut(df['nb_mots_brute'],
              bins=[-1, 0, 10, 30, 60, 100, df['nb_mots_brute'].max() + 1],
              labels=bins_labels)
cut_counts = cuts.value_counts().reindex(bins_labels)
axes[1].bar(bins_labels, cut_counts.values,
            color=sns.color_palette('YlOrRd', 6))
axes[1].set_title('Catégories de richesse textuelle', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nb mots')
axes[1].set_ylabel('Nombre de noms')
for p, v in zip(axes[1].patches, cut_counts.values):
    axes[1].text(p.get_x() + p.get_width()/2, v + 20, f'{v:,}',
                 ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Bag-of-Words — `origine_brute` et `origine` (lemmatisée)

In [ ]:
STOPWORDS = set([
    'de', 'du', 'des', 'le', 'la', 'les', 'un', 'une', 'et', 'en', 'au', 'aux',
    'il', 'elle', 'ils', 'elles', 'se', 'sa', 'son', 'ses', 'qui', 'que', 'qu',
    'ce', 'cet', 'cette', 'ces', 'par', 'pour', 'sur', 'dans', 'avec', 'est',
    'sont', 'a', 'à', 'été', 'être', 'plus', 'très', 'ou', 'si', 'ne',
    'pas', 'mais', 'car', 'dont', 'on', 'nous', 'vous', 'leur', 'leurs', 'tout',
    'aussi', 'comme', 'même', 'bien', 'puis', 'donc', 'les', 'celui', 'ceux',
    'nom', 'noms', 'forme', 'formes', 'var', 'porte', 'portent',
    's', 'd', 'l', 'j', 'n', 'y', 'c'
])

def tokenize(text):
    if not isinstance(text, str):
        return []
    tokens = re.findall(r"[a-zA-ZÀ-ÿ]{3,}", text.lower())
    return [t for t in tokens if t not in STOPWORDS]

# BoW sur origine_brute
bow_brute = Counter()
for text in df['origine_brute']:
    bow_brute.update(tokenize(str(text) if text else ''))

# BoW sur origine (lemmatisée)
bow_origine = Counter()
for text in df['origine']:
    bow_origine.update(tokenize(str(text) if text else ''))

print(f'Vocabulaire origine_brute (après stopwords) : {len(bow_brute):,} tokens uniques')
print(f'Vocabulaire origine       (après stopwords) : {len(bow_origine):,} tokens uniques')
print('\nTop 20 origine_brute :')
pd.DataFrame(bow_brute.most_common(20), columns=['mot', 'fréquence'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, bow, title, color in zip(
    axes,
    [bow_brute, bow_origine],
    ['Top 25 mots — origine_brute', 'Top 25 mots — origine (lemmatisée)'],
    ['#5b8fd6', '#7bcfa0']
):
    top = bow.most_common(25)
    ws, fs = zip(*top)
    ax.barh(list(ws)[::-1], list(fs)[::-1], color=color, alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Fréquence')
    for i, v in enumerate(list(fs)[::-1]):
        ax.text(v + 10, i, f'{v:,}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Bigrammes sur origine_brute
def bigrams(tokens):
    return [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]

bigram_counter = Counter()
for text in df['origine_brute']:
    toks = tokenize(str(text) if text else '')
    bigram_counter.update(bigrams(toks))

top_bi = bigram_counter.most_common(25)
bi_labels = [f'{a} {b}' for a, b in [x[0] for x in top_bi]]
bi_freqs  = [x[1] for x in top_bi]

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(bi_labels[::-1], bi_freqs[::-1],
        color=sns.color_palette('plasma', 25)[::-1])
ax.set_title('Top 25 bigrammes (origine_brute)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fréquence')
for i, v in enumerate(bi_freqs[::-1]):
    ax.text(v + 10, i, f'{v:,}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 6. Détection des langues / origines géographiques

In [ ]:
LANG_KEYWORDS = {
    'germanique':  ['germanique', 'germaniques', 'germain', 'allemand', 'frankish'],
    'français':    ['français', 'francais', 'française', 'vieux français', 'moyen français'],
    'latin':       ['latin', 'latine', 'latins', 'romano', 'gallo-romain', 'gallo-roman'],
    'breton':      ['breton', 'bretonne', 'bretons', 'celtique', 'celte'],
    'occitan':     ['occitan', 'occitane', 'languedocien', 'gascon', 'provençal'],
    'basque':      ['basque', 'euskara', 'euskal'],
    'flamand':     ['flamand', 'flamande', 'néerlandais', 'néerland'],
    'corse':       ['corse', 'corses'],
    'arabe':       ['arabe', 'arabes', 'arabo', 'arabique'],
    'hébreu':      ['hébreu', 'hébraïque', 'hébreux', 'bible'],
    'alsacien':    ['alsacien', 'alsacienne', 'alémanique'],
    'espagnol':    ['espagnol', 'espagnole', 'castillan', 'ibérique'],
    'italien':     ['italien', 'italienne', 'italiens'],
    'polonais':    ['polonais', 'polonaise', 'polonaises'],
    'anglais':     ['anglais', 'anglaise', 'anglaises', 'anglonormand'],
}

def detect_langs(text):
    if not isinstance(text, str):
        return []
    t = text.lower()
    return [lang for lang, kws in LANG_KEYWORDS.items() if any(kw in t for kw in kws)]

lang_counter = Counter()
multilang    = 0
none_detected = 0

for text in df['origine_brute']:
    langs = detect_langs(str(text) if text else '')
    for l in langs:
        lang_counter[l] += 1
    if len(langs) > 1:
        multilang += 1
    elif len(langs) == 0:
        none_detected += 1

print(f'Noms avec plusieurs origines détectées : {multilang:,} ({100*multilang/total:.1f}%)')
print(f'Noms sans origine détectée             : {none_detected:,} ({100*none_detected/total:.1f}%)')

lang_df = pd.DataFrame(lang_counter.most_common(), columns=['langue', 'nb_noms'])
lang_df['pct'] = (lang_df['nb_noms'] / total * 100).round(1)
print('\n', lang_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
palette_lang = sns.color_palette('tab20', len(lang_df))

# Bar
axes[0].barh(lang_df['langue'][::-1], lang_df['nb_noms'][::-1], color=palette_lang[::-1])
axes[0].set_title('Noms de famille par origine linguistique', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nombre de noms')
for i, v in enumerate(lang_df['nb_noms'][::-1]):
    axes[0].text(v + 20, i, f'{v:,}', va='center', fontsize=9)

# Pie
axes[1].pie(lang_df['nb_noms'], labels=lang_df['langue'],
            autopct='%1.1f%%', colors=palette_lang, startangle=140,
            textprops={'fontsize': 9})
axes[1].set_title('Parts relatives', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Mots-clés géographiques (départements, régions)
GEO_KEYWORDS = [
    'bretagne', 'normandie', 'alsace', 'lorraine', 'provence', 'occitanie',
    'gascogne', 'flandre', 'picardie', 'bourgogne', 'languedoc', 'savoie',
    'auvergne', 'vendée', 'anjou', 'touraine', 'poitou', 'berry', 'limousin',
    'aquitaine', 'pays basque', 'catalogne', 'corse', 'rhône', 'loire',
    'seine', 'garonne', 'pyrénée', 'alpes', 'ardenne', 'champagne'
]

geo_counter = Counter()
for text in df['origine_brute']:
    t = str(text).lower() if text else ''
    for kw in GEO_KEYWORDS:
        if kw in t:
            geo_counter[kw] += 1

geo_df = pd.DataFrame(geo_counter.most_common(20), columns=['région', 'nb_noms'])

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(geo_df['région'][::-1], geo_df['nb_noms'][::-1],
        color=sns.color_palette('YlGn', len(geo_df))[::-1])
ax.set_title('Références géographiques dans les origines (Top 20)', fontsize=12, fontweight='bold')
ax.set_xlabel('Nombre de noms')
for i, v in enumerate(geo_df['nb_noms'][::-1]):
    ax.text(v + 5, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 7. Analyse du graphe de noms liés (`noms_lies`)

In [ ]:
df['nb_lies'] = df['noms_lies'].apply(lambda x: len(x) if isinstance(x, list) else 0)

print('Distribution du nombre de noms liés :')
print(df['nb_lies'].describe().round(2))

n_isolated  = (df['nb_lies'] == 0).sum()
n_connected = (df['nb_lies'] > 0).sum()
print(f'\nNoms isolés (0 lien)   : {n_isolated:,} ({100*n_isolated/total:.1f}%)')
print(f'Noms connectés (≥1)    : {n_connected:,} ({100*n_connected/total:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie isolés vs connectés
axes[0].pie(
    [n_isolated, n_connected],
    labels=['Isolés', 'Connectés'],
    autopct='%1.1f%%',
    colors=['#aaa', '#5b8fd6'],
    startangle=90
)
axes[0].set_title('Noms isolés vs connectés', fontsize=12, fontweight='bold')

# Distribution nb de liens (noms connectés)
connected = df[df.nb_lies > 0]['nb_lies']
axes[1].hist(connected, bins=range(1, connected.max() + 2),
             color='#5b8fd6', edgecolor='none', alpha=0.85)
axes[1].set_title('Distribution du nb de noms liés (noms connectés)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nombre de noms liés')
axes[1].set_ylabel('Nombre de noms')
axes[1].set_yscale('log')
axes[1].set_ylabel('Nombre de noms (log)')

plt.tight_layout()
plt.show()

In [ ]:
# Top noms les plus connectés (hubs)
top_hubs = df.nlargest(20, 'nb_lies')[['nom', 'nb_lies', 'origine_brute']]
print('Top 20 noms avec le plus de variantes liées :')
display(top_hubs[['nom', 'nb_lies']].reset_index(drop=True))

## 8. Analyse des groupes (`id_groupe`)

In [ ]:
group_sizes = df['id_groupe'].value_counts()

print(f'Nombre de groupes distincts   : {len(group_sizes):,}')
print(f'Noms en singleton (groupe=1)  : {(group_sizes == 1).sum():,} ({100*(group_sizes==1).sum()/len(group_sizes):.1f}%)')
print(f'Taille max d\'un groupe        : {group_sizes.max()}')
print(f'Taille moyenne de groupe      : {group_sizes.mean():.2f}')
print(f'\nTop 10 groupes les plus grands :')

top_groups = group_sizes.head(10).reset_index()
top_groups.columns = ['id_groupe', 'taille']
# Ajouter un exemple de nom dans chaque groupe
top_groups['exemple'] = top_groups['id_groupe'].apply(
    lambda gid: df[df.id_groupe == gid]['nom_original'].iloc[0]
)
display(top_groups)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des tailles de groupes
axes[0].hist(group_sizes.values, bins=range(1, min(group_sizes.max() + 2, 30)),
             color='#a07bcf', edgecolor='none', alpha=0.85)
axes[0].set_title('Distribution des tailles de groupes', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Taille du groupe (nb noms)')
axes[0].set_ylabel('Nombre de groupes')
axes[0].set_yscale('log')

# Catégories
bins_g = [0, 1, 2, 5, 10, 20, group_sizes.max() + 1]
labels_g = ['singleton', '2', '3-5', '6-10', '11-20', '21+']
cats = pd.cut(group_sizes, bins=bins_g, labels=labels_g, right=True)
cat_counts = cats.value_counts().reindex(labels_g)

axes[1].bar(labels_g, cat_counts.values,
            color=sns.color_palette('Purples_r', len(labels_g)))
axes[1].set_title('Groupes par catégorie de taille', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Taille du groupe')
axes[1].set_ylabel('Nombre de groupes')
for p, v in zip(axes[1].patches, cat_counts.values):
    axes[1].text(p.get_x() + p.get_width()/2, v + 10, f'{v:,}',
                 ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 9. Word Cloud
> *Nécessite `wordcloud` : `pip install wordcloud`*

In [ ]:
try:
    from wordcloud import WordCloud

    # Word cloud global sur origine_brute
    all_text = ' '.join(str(t) for t in df['origine_brute'] if t)

    wc = WordCloud(
        width=1200, height=500,
        background_color='white',
        stopwords=STOPWORDS,
        max_words=200,
        colormap='plasma',
        collocations=False
    ).generate(all_text)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud — origine_brute', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Word cloud sur origine lemmatisée
    all_text_orig = ' '.join(str(t) for t in df['origine'] if t)

    wc2 = WordCloud(
        width=1200, height=500,
        background_color='#1a1a2e',
        stopwords=STOPWORDS,
        max_words=200,
        colormap='YlGn',
        collocations=False
    ).generate(all_text_orig)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(wc2, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud — origine (lemmatisée)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

except ImportError:
    print('⚠️  wordcloud non installé. Exécuter : pip install wordcloud')

## 10. Analyse de la diversité lexicale (TTR)

In [ ]:
# Type-Token Ratio par nom (richesse lexicale individuelle)
def ttr(text):
    if not isinstance(text, str) or not text.strip():
        return None
    tokens = tokenize(text)
    if not tokens:
        return None
    return len(set(tokens)) / len(tokens)

df['ttr'] = df['origine_brute'].apply(ttr)

print('Type-Token Ratio (richesse lexicale) :')
print(df['ttr'].describe().round(3))

fig, ax = plt.subplots(figsize=(10, 4))
ttr_vals = df['ttr'].dropna()
ax.hist(ttr_vals, bins=40, color='#f0a060', edgecolor='none', alpha=0.85)
ax.axvline(ttr_vals.median(), color='red', linestyle='--', linewidth=2,
           label=f'Médiane = {ttr_vals.median():.2f}')
ax.set_title('Distribution du TTR (Type-Token Ratio) — origine_brute',
             fontsize=12, fontweight='bold')
ax.set_xlabel('TTR (1 = lexique très varié)')
ax.set_ylabel('Nombre de noms')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Récapitulatif statistiques clés

In [ ]:
n_groupes  = df['id_groupe'].nunique()
singleton  = (group_sizes == 1).sum()

print('='*58)
print('  RÉCAPITULATIF — 1_noms_clean.json')
print('='*58)
print(f'  Nombre total de noms          : {total:>10,}')
print(f'  Longueur moy. des noms        : {df.longueur.mean():>10.1f} chars')
print(f'  Noms avec origine_brute       : {(df.nb_mots_brute > 0).sum():>10,}')
print(f'  Nb mots moy. (origine_brute)  : {df[df.nb_mots_brute>0].nb_mots_brute.mean():>10.1f}')
print(f'  Noms avec noms_lies           : {n_connected:>10,} ({100*n_connected/total:.1f}%)')
print(f'  Nb groupes (id_groupe)        : {n_groupes:>10,}')
print(f'  Groupes singletons            : {singleton:>10,} ({100*singleton/n_groupes:.1f}%)')
print(f'  Taille max d\'un groupe        : {group_sizes.max():>10}')
print(f'  Vocabulaire BoW origine_brute : {len(bow_brute):>10,}')
print(f'  Vocabulaire BoW origine       : {len(bow_origine):>10,}')
print('='*58)